In [1]:
import json
import gzip
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime

from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

## DuckDB reading of data (credit to Ilya M)

In [2]:
DATA_DIR = Path("../data/raw")
CATEGORY = "Musical_Instruments"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"

OUTPUT_DIR = Path("../data/processed")
OUTPUT_FILE  = OUTPUT_DIR / f"{CATEGORY}_merged.parquet"


In [3]:
c2 = duckdb.connect()

In [4]:
# Reviews File
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,"Great headphones, comfortable and sound is goo...",[],B003LPTAYI,B003LPTAYI,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650586000,0,True
1,3.0,nice sound. pedal failed after less than 1 year,I like the piano.. but the sustain pedal faile...,[],B00723436A,B06XP6TDVY,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1558567365290,2,True
2,4.0,okay,pretty good overall. I like it. the controll...,[],B0040FJ27S,B0040FJ27S,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1384912482000,0,True
3,3.0,Easy to return.,Too bad it didn't work. At least the return pr...,[],B00191WVF6,B00WJ3HL5I,AEM663T6XHZFWLODF4US2RCOCUSA,1607055693671,0,True
4,5.0,Good product despite tight bolt.,Good and sturdy but the bolt was hell to get o...,[],B07T9NM5QR,B07T9NM5QR,AFJTRBXMURLHS5EGNXLUHDHIZRFQ,1622595785255,0,False


In [5]:
# Metadata Files
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Musical Instruments,Pearl Export Lacquer EXL725S/C249 5-Piece New ...,4.2,22,[Item may ship in more than one box and may ar...,[Introducing the best selling drum set of all ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Best Selling Drum Set of All Time'...,Pearl,"[Musical Instruments, Drums & Percussion, Drum...","{'Item Weight': '""33 pounds""', 'Product Dimens...",B01M4HO6RK,None,None,<NA>
1,Musical Instruments,Behringer EUROPOWER EPQ900 Professional 900 Wa...,4.0,13,[2 x 390 Watts into 4 Ohms; 2 x 245 Watts into...,"[BEHRINGER EUROPOWER EPQ900, Professional 900-...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Behringer,"[Musical Instruments, Live Sound & Stage, Powe...","{'Item Weight': '""10.8 pounds""', 'Product Dime...",B00508JFE4,None,None,<NA>
2,Musical Instruments,Washburn Classical Series Acoustic Electric Cu...,3.6,15,[The Washburn is truly a professional instrume...,[C64SCE CLASSICAL GUITAR The cutaway allows ac...,399.0,[{'thumb': 'https://m.media-amazon.com/images/...,[],Washburn,"[Musical Instruments, Guitars]","{'Item Weight': '""5.98 pounds""', 'Product Dime...",B000S5JGMU,None,None,<NA>
3,Musical Instruments,"VocoPro, plug in, Black, 21.00 x 21.00 x 23.00...",3.5,7,"[Includes one microphone and one receiver, Can...",[VocoPro UHF-18 DIAMOND - N Wireless Microphon...,112.0,[{'thumb': 'https://m.media-amazon.com/images/...,[],VocoPro,"[Musical Instruments, Live Sound & Stage, PA S...","{'Item Weight': '""2.29 pounds""', 'Product Dime...",B00B2HLWZW,None,None,<NA>
4,Musical Instruments,Shure SM7B Vocal Dynamic Microphone for Broadc...,4.9,9512,[ONE MICROPHONE FOR EVERYTHING - Studio Record...,"[The SM7B dynamic microphone has a smooth, fla...",399.0,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Shure SM7B Mic Demonstration', 'ur...",Shure,"[Musical Instruments, Microphones & Accessorie...","{'Item Weight': '""2.7 pounds""', 'Product Dimen...",B0B89ZSYS7,None,None,<NA>


In [6]:
# Reviews schema
review_schema = c2.execute(f"""
    DESCRIBE SELECT * FROM read_json_auto('{REVIEWS_URL}')
""").df()
print("=== REVIEWS SCHEMA ===")
print(review_schema.to_string(index=False))

=== REVIEWS SCHEMA ===
      column_name                                                                                                   column_type null  key default extra
           rating                                                                                                        DOUBLE  YES None    None  None
            title                                                                                                       VARCHAR  YES None    None  None
             text                                                                                                       VARCHAR  YES None    None  None
           images STRUCT(small_image_url VARCHAR, medium_image_url VARCHAR, large_image_url VARCHAR, attachment_type VARCHAR)[]  YES None    None  None
             asin                                                                                                       VARCHAR  YES None    None  None
      parent_asin                                                

In [7]:
# Reviews schema
meta_schema = c2.execute(f"""
    DESCRIBE SELECT * FROM read_json_auto('{META_URL}')
""").df()
print("=== MetaData Schema ===")
print(meta_schema.to_string(index=False))

=== MetaData Schema ===
    column_name                                                               column_type null  key default extra
  main_category                                                                   VARCHAR  YES None    None  None
          title                                                                   VARCHAR  YES None    None  None
 average_rating                                                                    DOUBLE  YES None    None  None
  rating_number                                                                    BIGINT  YES None    None  None
       features                                                                 VARCHAR[]  YES None    None  None
    description                                                                 VARCHAR[]  YES None    None  None
          price                                                                      JSON  YES None    None  None
         images STRUCT(thumb VARCHAR, "large" VARCHAR, variant V

In [8]:
size_stats = c2.execute(f"""
    SELECT
        (SELECT COUNT(*) FROM read_json_auto('{REVIEWS_URL}')) AS total_reviews,
        (SELECT COUNT(*) FROM read_json_auto('{META_URL}'))    AS total_products,
        (SELECT COUNT(DISTINCT parent_asin) FROM read_json_auto('{REVIEWS_URL}')) AS unique_products_reviewed
""").df()
print(size_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_reviews  total_products  unique_products_reviewed
0        3017439          213593                    213571


## Selection and Justification of fields for retrieval

Based on the possible fields, we decided to keep the following fields in the dataframe because they have the most semantically rich information that would be useful for querying.  From EDA, there is no NA values as well which implies data quality for analysis. 

| File | Field | Type | Description |
|---|---|---|---|
| Review | `rating` | float | 1.0–5.0 star rating |
| Review | `title` | str | Review headline |
| Review | `text` | str | Full review body |
| Review | `parent_asin` | str | Parent product ID (use to join metadata) |
| Review | `verified_purchase` | bool | Verified buyer flag |
| Review | `helpful_vote` | int | Upvotes on the review |
| Meta | `title` | str | Product name |
| Meta | `average_rating` | float | Aggregate product rating |
| Meta | `features` | list[str] | Bullet-point features |
| Meta | `description` | list[str] | Product description paragraphs |
| Meta | `parent_asin` | str | Join key to reviews |

## Join the Datasets & Save to Processed Subfolder

In [14]:
if not OUTPUT_FILE.exists():
    c2.execute(f"""
        COPY (
            SELECT
                r.rating,
                r.title          AS review_title,
                r.text,
                r.parent_asin,
                r.verified_purchase,
                r.helpful_vote,
                m.title          AS product_title,
                m.average_rating,
                m.features,
                m.description
            FROM read_json_auto('{REVIEWS_URL}') r
            LEFT JOIN read_json_auto('{META_URL}') m
                ON r.parent_asin = m.parent_asin
        ) TO '{OUTPUT_FILE}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print(f"Saved to {OUTPUT_FILE}")
else:
    print(f"File already exists, skipping: {OUTPUT_FILE}")

File already exists, skipping: ..\data\processed\Musical_Instruments_merged.parquet


In [10]:
df = pd.read_parquet(f"{OUTPUT_FILE}")
df.head()

,rating,review_title,text,parent_asin,verified_purchase,helpful_vote,product_title,average_rating,features,description
0,3.0,I wouldn't know,I got this and they seem straight forward to i...,B09ZYNG4MK,True,0,Canomo 6 Pieces Sealed Guitar Tuning Pegs Keys...,4.4,"[Package includes: 6 pieces (3 pieces right, 3...",[Specification Guitar tuners pegs have quality...
1,5.0,Love it,Beautiful,B076DS2HZF,True,0,Deekec Zelda Ocarina 12 Hole Alto C with Song ...,4.4,"[ALL-IN-ONE: Comes with neck-strap cord,a lege...",[]
2,5.0,Great,Absolutely love it the only issue is mine pick...,B09JTBPNDM,True,0,Razer Seiren Mini USB Streaming Microphone: Pr...,4.6,[The #1 Best-Selling Gaming Peripherals Manufa...,[]
3,3.0,Come off,"They come off, I mean they just slip off and w...",B08GLS23Q9,True,1,Piano Keyboard Stickers Removable for Beginner...,4.5,[▶ This piano key sticker is suitable for almo...,[]
4,5.0,Listen to this.,Good quality of sound. Good response. I have t...,B01AJ2UIIC,True,0,"DS18 PRO-NEO8 Loudspeaker - 8"", Midrange, Heav...",4.4,"[8"" LOUDSPEAKER WITH BULLET - Specifically des...","[DS18 PRO 8"" NEODINIUM MIDRANGE 4OHM]"


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3017439 entries, 0 to 3017438
Data columns (total 10 columns):
 #   Column             Dtype  
---  ------             -----  
 0   rating             float64
 1   review_title       str    
 2   text               str    
 3   parent_asin        str    
 4   verified_purchase  bool   
 5   helpful_vote       int64  
 6   product_title      str    
 7   average_rating     float64
 8   features           object 
 9   description        object 
dtypes: bool(1), float64(2), int64(1), object(2), str(4)
memory usage: 1.3+ GB


In [12]:
df.isna().sum()

rating               0
review_title         0
text                 0
parent_asin          0
verified_purchase    0
helpful_vote         0
product_title        0
average_rating       0
features             0
description          0
dtype: int64

## Preprocessing

In [13]:
import re

def preprocess_for_search(text: str) -> str:
    """
    Clean text for BM25 or semantic search modelling.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # 1. Lowercase
    text = text.lower()

    # 2. Expand common contractions
    contractions = {
        "won't": "will not", "can't": "cannot", "n't": " not",
        "'re": " are", "'ve": " have", "'ll": " will",
        "'d": " would", "'m": " am"
    }
    for k, v in contractions.items():
        text = text.replace(k, v)

    # 3. Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # 4. Remove URLs and emails
    text = re.sub(r"http\S+|www\.\S+|\S+@\S+", " ", text)

    # 5. Remove non-alphanumeric characters (keep spaces)
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    return text

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    string_cols = df.select_dtypes(include="object").columns
    
    for col in string_cols:
        df[col] = df[col].apply(lambda x: preprocess_for_search(x))
    
    return df

## Description of text preprocessing decisions:

As shown in the function above, we are cleaning the corpus to:
- convert all text to lower case
- convert common contractions to full words
- remove html, URLs, emails, and non-alphanumeric characters. 

This will allow our models to optimally understand and identify our corpus. 